# Ball Detection — Hough Circles

## Imports

In [ ]:
import os
import cv2
import math
import numpy as np
import matplotlib.pyplot as plt

## Load Dataset

In [ ]:
DATASET_DIR = "development_set/"

In [ ]:
image_paths = sorted([
    os.path.join(DATASET_DIR, f)
    for f in os.listdir(DATASET_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

print(f"Found {len(image_paths)} images")

In [ ]:
def show_images(images, titles=None, max_cols=4, figsize_per_image=(4, 3)):
    n = len(images)
    if n == 0:
        print("No images to display.")
        return

    cols = min(n, max_cols)
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(figsize_per_image[0] * cols,
                                                   figsize_per_image[1] * rows))
    axes = np.array(axes).flatten()

    for i, ax in enumerate(axes):
        if i < n:
            img = images[i]
            if isinstance(img, str):
                img = cv2.imread(img)

            if img is not None:
                if img.ndim == 2:                          # ← grayscale / mask
                    ax.imshow(img, cmap='gray', vmin=0, vmax=255)
                else:                                      # ← colour image
                    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

            ax.set_title(titles[i] if titles and i < len(titles) else f"Image {i+1}",
                         fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## Ball Detection

In [ ]:
def preprocess_lighting(hsv_image):
    # 1. Split the HSV image into its three separate channels
    h, s, v = cv2.split(hsv_image)

    # 2. Create the CLAHE filter
    # clipLimit prevents noise from being amplified too much
    # tileGridSize is the size of the localized "checkerboard" squares
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    # 3. Apply the filter ONLY to the V (brightness) channel
    v_eq = clahe.apply(v)

    # 4. Merge the channels back together
    hsv_eq = cv2.merge((h, s, v_eq))

    return hsv_eq

def get_table_mask(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hsv = preprocess_lighting(hsv)
    h, w = image.shape[:2]

    # Sample using median to avoid balls in the center
    # 1. Define 5 safe sampling points (Center + 4 inner quadrants)
    points = [
        (h//2, w//2),               # Center
        (int(h*0.4), int(w*0.4)),   # Top-Left inner
        (int(h*0.4), int(w*0.6)),   # Top-Right inner
        (int(h*0.6), int(w*0.4)),   # Bottom-Left inner
        (int(h*0.6), int(w*0.6))    # Bottom-Right inner
    ]

    # 2. Collect a 40x40 patch from ALL 5 locations
    samples = []
    for py, px in points:
        patch = hsv[py-20:py+20, px-20:px+20]
        samples.append(patch)

    # 3. Stack all 8,000 pixels together and find the true median
    all_samples = np.vstack(samples)
    median_hsv = np.median(all_samples, axis=(0, 1))

    # Build tolerance and threshold
    tol = np.array([15, 80, 80])
    lower = np.clip(median_hsv - tol, 0, 255).astype(np.uint8)
    upper = np.clip(median_hsv + tol, 0, 255).astype(np.uint8)

    return cv2.inRange(hsv, lower, upper)

def isolate_largest_blob(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return np.zeros_like(mask)

    largest_contour = max(contours, key=cv2.contourArea)

    clean_mask = np.zeros_like(mask)

    cv2.drawContours(clean_mask, [largest_contour], -1, 255, thickness=cv2.FILLED)

    return clean_mask

In [ ]:
def get_ball_circles(img):
    """
    Receives a BGR image array.
    Returns a list of circles in the format: [(center_x, center_y, radius), ...]
    """
    # Get the table mask using your median sampling function
    table_mask = get_table_mask(img)

    # Get the solid playing area using your blob isolation function
    playing_area_mask = isolate_largest_blob(table_mask)

    # The balls are everything INSIDE the playing area that is NOT the table color.
    not_table_mask = cv2.bitwise_not(table_mask)
    balls_mask = cv2.bitwise_and(not_table_mask, playing_area_mask)

    # Clean up the balls mask
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    balls_mask = cv2.morphologyEx(balls_mask, cv2.MORPH_OPEN, kernel, iterations=1)

    # Find contours of the balls
    ball_contours, _ = cv2.findContours(balls_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    circles = []

    for cnt in ball_contours:
        area = cv2.contourArea(cnt)
        if 50 < area < 4000:
            # We still calculate the bounding box just to check the aspect ratio
            # to ensure the object is roughly square/circular (filters out long sticks)
            x, y, w, h = cv2.boundingRect(cnt)
            aspect_ratio = float(w) / h

            if 0.5 < aspect_ratio < 1.8:
                # Calculate the minimum enclosing circle for the contour
                (center_x, center_y), radius = cv2.minEnclosingCircle(cnt)

                # Convert to integers as pixel coordinates cannot be decimals
                circles.append((int(center_x), int(center_y), int(radius)))

    return circles

In [ ]:
processed_images = []
image_titles = []

print(f"Processing {len(image_paths)} images. This might take a few seconds...")

for path in image_paths:
    # Read the image outside the function
    img = cv2.imread(path)

    if img is not None:
        # 1. Get ONLY the circle coordinates from the function
        circles = get_ball_circles(img)

        # 2. Draw the circles AFTERWARD on a copy of the image
        result_img = img.copy()
        for (cx, cy, r) in circles:
            # Draw the circle outline (Red in BGR format is (0, 0, 255))
            cv2.circle(result_img, (cx, cy), r, (0, 0, 255), 3)

            # Optional: Draw a tiny dot in the exact center of the ball
            cv2.circle(result_img, (cx, cy), 1, (0, 255, 0), 2)

        processed_images.append(result_img)

        # Create a nice title with the filename and the number of balls found
        filename = os.path.basename(path)
        image_titles.append(f"{filename} ({len(circles)} balls)")


# --- 3. Display everything using your function ---
show_images(processed_images, titles=image_titles, max_cols=3, figsize_per_image=(8, 6))